In [ ]:
import scanpy as sc
import anndata as ad
import scanpy.external as sce

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scvi
import scipy.sparse as sp

from rich import print
import warnings
warnings.filterwarnings("ignore")
import os

outdir = "/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA"
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = "/scratch/user/s4575250/BIOX7018_thesis/figures/PICA/03_abundance_change"

In [ ]:
# load annotations
adata = sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/data/.h5ad")

adata.obs['cell_type'].unique()
adata.obs['cell_type'].value_counts()

In [ ]:
adata.obs["pica_id"].unique()
adata.obs['pica_id'].value_counts()

### load metadata

In [ ]:
meta_raw = pd.read_csv("/scratch/user/s4575250/BIOX7018_thesis/data/PICA-PICAReportOverall_DATA_LABELS_2025-07-25_1224.csv", index_col=0)

In [ ]:
print(meta_raw["IFC ID"])

In [ ]:
# convert IFC ID -> PICA format (PICA0001)
meta = meta_raw.copy()
meta["IFC ID"] = meta["IFC ID"].fillna("0").astype(str)
meta["IFC ID"] = meta["IFC ID"].astype(str)

meta["pica_id"] = (
    meta["IFC ID"]
    .str.extract(r"(\d+)")[0]
    .fillna("0")
    .astype(int)
    .apply(lambda x: f"PICA{x:04d}")
)
meta.set_index("pica_id", inplace=True)

print(meta["IFC ID"])

In [ ]:
# restrict meta to only those in adata
adata_picas = adata.obs["pica_id"].unique()
meta_sub = meta.loc[meta.index.isin(adata_picas)]
adata_sub = adata[adata.obs["pica_id"].isin(meta_sub.index)].copy()

print(f"Metadata before: {meta.shape[0]} rows")
print(f"Metadata after subsetting: {meta_sub.shape[0]} rows")
print(f"AnnData before: {adata.shape[0]} cells")
print(f"AnnData after subsetting: {adata_sub.shape[0]} cells")

In [ ]:
# join metadata to adata
adata_with_meta = adata_sub.copy()

adata_with_meta.obs["pica_id"] = adata_with_meta.obs["pica_id"].astype("string")
adata_with_meta.obs = adata_with_meta.obs.join(meta_sub, on="pica_id", how="left", rsuffix="_meta")

In [ ]:
print("Unique donors in metadata subset:", meta_sub.index.unique().tolist())
print("Unique donors in adata_with_meta:", adata_with_meta.obs['pica_id'].unique().tolist())


In [ ]:
meta_summary = (
    adata_with_meta.obs[["pica_id", "Child's sex", "Age in years at the time of collection"]]
    .drop_duplicates()
    .sort_values("pica_id")
)
print(meta_summary)

In [ ]:
meta_df = adata_with_meta.obs[['pica_id', "Child's sex", "Age in years at the time of collection"]].drop_duplicates()

plt.figure(figsize=(5,5))
sns.boxplot(data=meta_df, x="Child's sex", y="Age in years at the time of collection", palette="Set2")
sns.stripplot(data=meta_df, x="Child's sex", y="Age in years at the time of collection", color="k", size=3, alpha=0.6)
plt.title("Age distribution by sex (per donor)")
plt.ylabel("Age (years)")
plt.xlabel("Sex")
plt.show()

In [ ]:
adata_with_meta.obs = adata_with_meta.obs.applymap(
    lambda x: str(x) if not isinstance(x, (int, float, str)) else x
)

In [ ]:
adata_with_meta.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_meta.h5ad", compression="gzip")

### clean up age

In [ ]:
adata_age=sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_meta.h5ad")

In [ ]:
adata_age.obs.groupby("pica_id")["Age in years at the time of collection"].unique()

In [ ]:
adata_age.obs["Age in years int"] = (
    adata_age.obs["Age in years at the time of collection"]
    .astype(float)
    .apply(np.floor)  
    .astype(int)
)

In [ ]:
adata_age.obs.groupby("pica_id")["Age in years int"].unique()

In [ ]:
# Define bins and labels
bins = [0, 3, 6, 10, 14, 18]
labels = ["0–3", "4–6", "7–10", "11–14", "15–18"]

# Create new column for grouped ages
adata_age.obs["Age_group"] = pd.cut(
    adata_age.obs["Age in years int"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

In [ ]:
adata_age.obs["Age_group"].value_counts().sort_index()

In [ ]:
adata_age.obs["Age_group"].value_counts().sort_index().plot(
    kind="bar", figsize=(6,4), color="lightsteelblue"
)
plt.title("Cell distribution across age groups (0–18 years)")
plt.xlabel("Age group")
plt.ylabel("Cell count")
plt.show()

In [ ]:
# create donor-level age group summary
age_meta_df = adata_age.obs[["pica_id", "Age_group"]].drop_duplicates()
age_meta_df["Age_group"].value_counts().sort_index()

In [ ]:
sc.pl.umap(adata_age, color=["Child's sex", "Age in years int", "Age_group"], wspace=0.4)

In [ ]:
sc.pl.umap(adata_age, color=["Child's sex", "Age in years int", "Age_group"], wspace=0.4)

In [ ]:
adata_age.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_with_age.h5ad", compression="gzip")

### extract CD4 annotations

In [ ]:
adata_age_cd4 = sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_with_age.h5ad")

In [ ]:
adata_age_cd4.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd4_combined_annot_with_age.h5ad", compression='gzip')

#### run scvi for cd4

In [ ]:
adata_scvi_cd4 =sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd4_combined_annot_with_age.h5ad")

In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi_cd4, layer="counts", batch_key="pica_id")

In [ ]:
model_cd4 = scvi.model.SCVI(adata_scvi_cd4, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
model_cd4.train()

In [ ]:
adata_scvi_cd4.obsm["X_scVI"] = model_cd4.get_latent_representation()

#  latent representation 
sc.pp.neighbors(adata_scvi_cd4, use_rep="X_scVI")
sc.tl.umap(adata_scvi_cd4)


sc.pl.umap(adata_scvi_cd4, color=[ "pica_id", 'cell_type'], save="_scvi_integrated_umap.png")

In [ ]:
adata_scvi_cd4.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd4_combined_annot_with_age_scvi.h5ad", compression="gzip")

In [ ]:
sc.pl.umap(
    adata_scvi_cd4,
    color=["log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
    save="_scvi_qc.png"
)

#### convert to sce object

In [ ]:
import anndata2ri
import rpy2.robjects as robjects
from rpy2.robjects.conversion import localconverter

with localconverter(robjects.default_converter + anndata2ri.converter):
    sce = robjects.conversion.py2rpy(adata_scvi_cd4)

# Save the sce object in .rds file
robjects.globalenv["sce"] = sce
robjects.r("saveRDS(sce, file='{}')".format("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd4_combined_annot_with_age_scvi.sce.rds"))

### extract CD8 annotations

In [ ]:
adata_age_cd8 = sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_with_age.h5ad")

In [ ]:
adata_age_cd4.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd8_combined_annot_with_age.h5ad",compression='gzip')

#### run scvi for cd8

In [ ]:
adata_scvi_cd8 =sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd8_combined_annot_with_age.h5ad")

In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi_cd8, layer="counts", batch_key="pica_id")

In [ ]:
model_cd8 = scvi.model.SCVI(adata_scvi_cd8, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
model_cd8.train()

In [ ]:
adata_scvi_cd8.obsm["X_scVI"] = model_cd8.get_latent_representation()

#  latent representation 
sc.pp.neighbors(adata_scvi_cd8, use_rep="X_scVI")
sc.tl.umap(adata_scvi_cd8)


sc.pl.umap(adata_scvi_cd8, color=[ "pica_id", 'cell_type'], save="_scvi_integrated_umap.png")

In [ ]:
adata_scvi_cd8.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd8_combined_annot_with_age_scvi.h5ad", compression="gzip")

In [ ]:
sc.pl.umap(
    adata_scvi_cd8,
    color=["log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
    save="_scvi_qc.png"
)

#### convert to sce object

In [ ]:
import anndata2ri
import rpy2.robjects as robjects
from rpy2.robjects.conversion import localconverter

with localconverter(robjects.default_converter + anndata2ri.converter):
    sce = robjects.conversion.py2rpy(adata_scvi_cd8)

# Save the sce object in .rds file
robjects.globalenv["sce"] = sce
robjects.r("saveRDS(sce, file='{}')".format("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_cd8_combined_annot_with_age_scvi.sce.rds"))

### extract nk annotations

In [ ]:
adata_age_nk = sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_combined_annot_with_age.h5ad")

In [ ]:
adata_age_cd4.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_nk_combined_annot_with_age.h5ad", compression='gzip')

#### run scvi for nk

In [ ]:
adata_scvi_nk =sc.read_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_nk_combined_annot_with_age.h5ad")

In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi_nk, layer="counts", batch_key="pica_id")

In [ ]:
model_nk = scvi.model.SCVI(adata_scvi_nk, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
model_nk.train()

In [ ]:
adata_scvi_nk.obsm["X_scVI"] = model_nk.get_latent_representation()

#  latent representation 
sc.pp.neighbors(adata_scvi_nk, use_rep="X_scVI")
sc.tl.umap(adata_scvi_nk)


sc.pl.umap(adata_scvi_nk, color=[ "pica_id", 'cell_type'], save="_scvi_integrated_umap.png")

In [ ]:
adata_scvi_nk.write_h5ad("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_nk_combined_annot_with_age_scvi.h5ad", compression="gzip")

In [ ]:
sc.pl.umap(
    adata_scvi_nk,
    color=["log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
    save="_scvi_qc.png"
)

#### convert to sce object

In [ ]:
import anndata2ri
import rpy2.robjects as robjects
from rpy2.robjects.conversion import localconverter

with localconverter(robjects.default_converter + anndata2ri.converter):
    sce = robjects.conversion.py2rpy(adata_scvi_nk)

# Save the sce object in .rds file
robjects.globalenv["sce"] = sce
robjects.r("saveRDS(sce, file='{}')".format("/scratch/user/s4575250/BIOX7018_thesis/write/03_batch_expression/PICA/PICA_nk_combined_annot_with_age_scvi.sce.rds"))